# C11-neural-training — Practice p24 — Solution


**Type:** challenge · **Difficulty:** advanced · **Concepts:** torch-optimizers, trained-mlp, batch-normalization, dropout


Each configuration begins from the same seeded construction point, snapshots
parameters and BatchNorm buffers, and lets Adam own only trainable parameters.
Mode probes are computed from the trained model rather than hard-coded flags.


In [ ]:
import torch
import torch.nn as nn

def _run_ablation_core(seed=20260804, epochs=250):
    torch.use_deterministic_algorithms(True)
    torch.set_default_dtype(torch.float64)
    g = torch.Generator(device="cpu").manual_seed(seed)
    centers = torch.tensor([[-2., -1.], [0., 2.], [2., -1.]], dtype=torch.float64)
    X = torch.cat([center + .30 * torch.randn((40, 2), generator=g, dtype=torch.float64) for center in centers])
    y = torch.arange(3).repeat_interleave(40)
    configs = {
        "full": (True, .25, False),
        "frozen_affine": (True, .25, True),
        "no_dropout": (True, 0., False),
        "no_batchnorm": (False, .25, False),
    }
    output = {}
    audits = {}
    for name, (use_bn, p_drop, freeze) in configs.items():
        torch.manual_seed(seed)
        layers = [nn.Linear(2, 12)]
        if use_bn:
            layers.append(nn.BatchNorm1d(12))
        layers.extend([nn.ReLU(), nn.Dropout(p_drop), nn.Linear(12, 3)])
        model = nn.Sequential(*layers).to(dtype=torch.float64, device="cpu")
        bn = next((module for module in model if isinstance(module, nn.BatchNorm1d)), None)
        if freeze:
            bn.weight.requires_grad_(False)
            bn.bias.requires_grad_(False)
        initial = {parameter_name: p.detach().clone() for parameter_name, p in model.named_parameters()}
        initial_buffers = None if bn is None else (bn.running_mean.clone(), bn.running_var.clone())
        owned_named = [(parameter_name, p) for parameter_name, p in model.named_parameters() if p.requires_grad]
        optimizer = torch.optim.Adam([p for _, p in owned_named], lr=.03)
        criterion = nn.CrossEntropyLoss()
        losses = torch.empty(epochs, dtype=torch.float64)
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X), y)
            losses[epoch] = loss.detach()
            loss.backward()
            optimizer.step()
        current = dict(model.named_parameters())
        bn_prefix = None if bn is None else str(list(model).index(bn))
        linear_names = tuple(
            parameter_name for parameter_name in current
            if bn_prefix is None or not parameter_name.startswith(bn_prefix + ".")
        )
        linear_movement = max(
            float(torch.linalg.vector_norm(current[parameter_name].detach() - initial[parameter_name]))
            for parameter_name in linear_names
        )
        if bn is None:
            affine_movement = None
            buffer_changed = None
            trained_buffers = None
        else:
            affine_names = (f"{bn_prefix}.weight", f"{bn_prefix}.bias")
            affine_movement = max(
                float(torch.linalg.vector_norm(current[parameter_name].detach() - initial[parameter_name]))
                for parameter_name in affine_names
            )
            trained_buffers = (bn.running_mean.clone(), bn.running_var.clone())
            buffer_changed = not all(
                torch.equal(before, after) for before, after in zip(initial_buffers, trained_buffers)
            )
        model.eval()
        with torch.no_grad():
            accuracy = float((model(X).argmax(1) == y).double().mean())
        model.train()
        torch.manual_seed(seed)
        train_a = model(X).detach()
        train_b = model(X).detach()
        train_equal = bool(torch.allclose(train_a, train_b, atol=1e-9, rtol=1e-7))
        model.eval()
        eval_buffers_before = None if bn is None else (bn.running_mean.clone(), bn.running_var.clone())
        with torch.no_grad():
            eval_a = model(X)
            eval_b = model(X)
        eval_buffers_after = None if bn is None else (bn.running_mean.clone(), bn.running_var.clone())
        eval_equal = bool(torch.allclose(eval_a, eval_b, atol=1e-9, rtol=1e-7))
        output[name] = {
            "initial_loss": float(losses[0]),
            "final_loss": float(losses[-1]),
            "accuracy": accuracy,
            "linear_movement": linear_movement,
            "bn_affine_movement": affine_movement,
            "bn_buffer_changed": buffer_changed,
            "optimizer_state_entries": len(optimizer.state),
            "train_repeat_equal": train_equal,
            "eval_repeat_equal": eval_equal,
            "model": model,
        }
        audits[name] = {
            "owned_parameter_names": tuple(parameter_name for parameter_name, _ in owned_named),
            "optimizer_parameter_names": tuple(
                next(parameter_name for parameter_name, candidate in owned_named if candidate is p)
                for group in optimizer.param_groups for p in group["params"]
            ),
            "optimizer_state_entries": len(optimizer.state),
            "step_counts": torch.tensor(
                [int(optimizer.state[p].get("step", torch.tensor(0)).item()) for _, p in owned_named],
                dtype=torch.int64,
            ),
            "final_grads": tuple(None if p.grad is None else p.grad.detach().clone() for _, p in owned_named),
            "initial_parameters": initial,
            "linear_names": linear_names,
            "bn_prefix": bn_prefix,
            "initial_buffers": initial_buffers,
            "trained_buffers": trained_buffers,
            "eval_buffers_before": eval_buffers_before,
            "eval_buffers_after": eval_buffers_after,
            "train_outputs": (train_a, train_b),
            "eval_outputs": (eval_a, eval_b),
        }
    return output, audits

def run_ablation(seed=20260804, epochs=250):
    public, _ = _run_ablation_core(seed=seed, epochs=epochs)
    return public

result_p24 = run_ablation()


### Answer check


In [ ]:
assert set(result_p24) == {"full", "frozen_affine", "no_dropout", "no_batchnorm"}
audited_public_p24, audits_p24 = _run_ablation_core()
for name, row in result_p24.items():
    audited_row = audited_public_p24[name]
    audit = audits_p24[name]
    assert set(row) == {
        "initial_loss", "final_loss", "accuracy", "linear_movement", "bn_affine_movement",
        "bn_buffer_changed", "optimizer_state_entries", "train_repeat_equal", "eval_repeat_equal", "model"
    }
    for key in (
        "initial_loss", "final_loss", "accuracy", "linear_movement", "bn_affine_movement",
        "bn_buffer_changed", "optimizer_state_entries", "train_repeat_equal", "eval_repeat_equal",
    ):
        assert row[key] == audited_row[key]
    for p, q in zip(row["model"].parameters(), audited_row["model"].parameters()):
        assert torch.allclose(p, q, atol=1e-9, rtol=1e-7)
    for left, right in zip(row["model"].buffers(), audited_row["model"].buffers()):
        assert torch.equal(left, right)

    owned = dict(audited_row["model"].named_parameters())
    assert audit["owned_parameter_names"] == audit["optimizer_parameter_names"]
    assert set(audit["owned_parameter_names"]) == {parameter_name for parameter_name, p in owned.items() if p.requires_grad}
    assert audit["optimizer_state_entries"] == len(audit["owned_parameter_names"]) == row["optimizer_state_entries"]
    assert torch.equal(audit["step_counts"], torch.full_like(audit["step_counts"], 250))
    assert all(
        grad is not None and grad.shape == owned[parameter_name].shape and torch.isfinite(grad).all()
        for parameter_name, grad in zip(audit["owned_parameter_names"], audit["final_grads"])
    )
    independent_linear_movement = max(
        float(torch.linalg.vector_norm(owned[parameter_name].detach() - audit["initial_parameters"][parameter_name]))
        for parameter_name in audit["linear_names"]
    )
    assert abs(independent_linear_movement - row["linear_movement"]) <= 1e-12
    assert row["final_loss"] < row["initial_loss"] and row["linear_movement"] > .05
    assert bool(torch.allclose(*audit["train_outputs"], atol=1e-9, rtol=1e-7)) == row["train_repeat_equal"]
    assert bool(torch.allclose(*audit["eval_outputs"], atol=1e-9, rtol=1e-7)) == row["eval_repeat_equal"]
    assert row["eval_repeat_equal"]
    if audit["bn_prefix"] is not None:
        affine_names = (f'{audit["bn_prefix"]}.weight', f'{audit["bn_prefix"]}.bias')
        independent_affine_movement = max(
            float(torch.linalg.vector_norm(owned[parameter_name].detach() - audit["initial_parameters"][parameter_name]))
            for parameter_name in affine_names
        )
        assert abs(independent_affine_movement - row["bn_affine_movement"]) <= 1e-12
        independent_buffer_change = not all(
            torch.equal(before, after)
            for before, after in zip(audit["initial_buffers"], audit["trained_buffers"])
        )
        assert independent_buffer_change == row["bn_buffer_changed"]
        assert all(
            torch.equal(before, after)
            for before, after in zip(audit["eval_buffers_before"], audit["eval_buffers_after"])
        )
assert result_p24["full"]["accuracy"] >= .95
assert result_p24["frozen_affine"]["bn_affine_movement"] == 0.0
assert result_p24["full"]["bn_affine_movement"] > 0.0
assert result_p24["full"]["bn_buffer_changed"]
assert result_p24["frozen_affine"]["bn_buffer_changed"]
assert result_p24["no_dropout"]["bn_buffer_changed"]
assert result_p24["no_batchnorm"]["bn_affine_movement"] is None
assert result_p24["no_batchnorm"]["bn_buffer_changed"] is None
assert not result_p24["full"]["train_repeat_equal"]
assert not result_p24["frozen_affine"]["train_repeat_equal"]
assert not result_p24["no_batchnorm"]["train_repeat_equal"]
assert result_p24["no_dropout"]["train_repeat_equal"]
